In [32]:
import numpy as np
datapath = "../../Preprocessing/Encoded_Data/ecoli_rna_embeddings.npz"
data = np.load(datapath)
embeddings = data["embeddings"]
labels = data["labels"]

In [36]:
counts = {}
for label in labels:
    counts[label] = counts[label] + 1 if label in counts else 1
print(counts)

{np.str_('rRNA'): 16, np.str_('tRNA'): 45, np.str_('sRNA'): 100}


In [43]:
mid_layer  = embeddings[:, 0, :]
last_layer = embeddings[:, 1, :]
embeddings_pooled = embeddings.mean(axis=1)
embeddings_pooled.shape

(161, 512)

In [46]:
print("Mid layer:", mid_layer[0][:10])
print("Last layer:", last_layer[0][:10])
print("Pooled: ", embeddings_pooled[0][:10])

Mid layer: [ 0.04351216 -1.0366505   0.84144366  1.6133971  -0.52261096  0.93100864
 -2.8923101  -0.44584432 -1.446854   -2.664081  ]
Last layer: [ 1.0053989  -0.8118206   0.636622    2.7043056  -0.29271957  0.8397843
 -2.9534838   1.4206294   2.317268    0.92349327]
Pooled:  [ 0.5244555  -0.9242356   0.73903286  2.1588514  -0.40766525  0.8853965
 -2.9228969   0.48739254  0.43520695 -0.8702939 ]


In [30]:
print(f"Number of clusters found: {len(set(labels)) - (1 if -1 in labels else 0)}")
print(f"Noise points: {sum(labels == -1)}")

Number of clusters found: 0
Noise points: 161


In [56]:
from sklearn.metrics import v_measure_score
from sklearn.cluster import HDBSCAN

best_score = 0
best_params = {}
best_results = {}
embedding_options = {"Mid layer": mid_layer, "Last layer": last_layer, "Pooled": embeddings_pooled}

for min_cluser_size in [5, 10, 20, 30, 50]:
    for min_samples in [1, 5, 10]:
        for metric in ["euclidean", "manhattan"]:
            for embedding in embedding_options:
                clusterer = HDBSCAN(
                min_cluster_size=min_cluser_size,
                min_samples=min_samples,
                metric=metric,
                copy=True
                )
                hdb_labels = clusterer.fit_predict(embedding_options[embedding])

                if len(set(hdb_labels)) == 1: # only noise
                    continue

                score = v_measure_score(labels, hdb_labels)

                if score > best_score:
                    best_score = score
                    best_params = {
                        'min_cluster_size': min_cluser_size,
                        'min_samples': min_samples,
                        'metric': metric,
                        'embedding': embedding
                    }
                    best_results = {
                        'Number of cluster': len(set(hdb_labels)),
                        'Number of noise points': sum(hdb_labels == -1)
                    }

print("Best score:", best_score)
print("Best parameters:", best_params)
print("Best result:", best_results)

Best score: 0.22243871176308355
Best parameters: {'min_cluster_size': 5, 'min_samples': 1, 'metric': 'euclidean', 'embedding': 'Mid layer'}
Best result: {'Number of cluster': 5, 'Number of noise points': np.int64(82)}
